# Coding Project 1: Linear Regression and Regularization

**Please write the names of all group members here:**

- *(name 1)*
- *(name 2)*




---

*Note:* The provided structure for the code below is only suggestive. You may structure your program differently, provided that your notebook is readable, reproducible, and answers all questions clearly.

A minimal `sklearn` workflow example is included before Question 3. It is intended to illustrate the use of `ColumnTransformer`, `Pipeline`, and `GridSearchCV`; it does **not** solve the project questions for you.

### Setup

All packages and all fixed constants of the analysis, in one place.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import sklearn
import statsmodels
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

SEED = 42          # fixed once, before looking at any result: used for the split, the CV folds and TruncatedSVD
TEST_SIZE = 0.30   # 70% training / 30% test (Question 1.c)
N_FOLDS = 8        # 8-fold cross-validation with the same folds for every method (Question 3.b)
DATA_FILE = "data/Housing.csv"
DESCRIPTION_FILE = "data/data_description.txt"

pd.set_option("display.max_columns", 60)
print(f"pandas {pd.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"scikit-learn {sklearn.__version__} | statsmodels {statsmodels.__version__}")

### Helper functions

Small reusable functions used in several questions (the assignment asks for them under *Code quality*).

In [ ]:
# Helper functions, each with a one-line docstring (plan: section 2 of helpers/Analysis/4_Plan_and_Decisions.md):
#   parse_description(path)      -> 35 numerical / 44 categorical names + listed levels
#   load_housing(path)           -> X, y read safely ('NA'/'None' kept as text, Id as index)
#   recode_missing(X)            -> 'NA' that is not a valid level -> NaN (stateless)
#   prepare_features(Xtr, Xte)   -> Question 1.d by hand (training statistics only)
#   regression_metrics(model, Xtr, Xte) -> MSE and R2, in/out of sample, log and USD scale
#   ols_matrix(A, y, method)     -> beta, sigma^2, SE with 'solve' or 'pinv' (Question 2.b)
#   make_preprocessor()          -> the Question 3.b.i component
#   tune(estimator, grid, svd)   -> GridSearchCV with the shared 8 folds
#   grid_edges(search, grid)     -> selected values on the edge of the grid?
#   cv_summary(search)           -> best params, mean CV MSE (sign flipped) and its std

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2:
#   Tested reference versions of parse_description, load_raw (= load_housing), recode_missing,
#   prepare_manual (= prepare_features) and make_preprocessor are in helpers/Analysis/scripts/helpers.py.
#   Adapt them here; the submitted notebook must not import from the Analysis folder.

### Question 1 - Importing and Preparing the Data

#### 1.a) Import the data and separate X and y

In [ ]:
# 1.a) Import Housing.csv as a pandas DataFrame.
# Separate the explanatory features X from the target y = SalePrice.
#
# Guidance:
# - Refer to data_description.txt to determine how variables should be interpreted.
# - The first column of the csv file is an index and should not be used as an explanatory variable.
# - Keep the original variable names so that later coefficient tables are easy to interpret.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 1.a:
#   pandas' default read_csv turns the strings 'NA' AND 'None' into NaN: the 864 valid 'None' levels of
#   MasVnrType and every valid 'NA' level would be lost. Read with keep_default_na=False and set
#   na_values only for the 35 numerical columns.
#   Take the 35 numerical / 44 categorical lists from data_description.txt, not from the dtypes:
#   MSSubClass is stored as integers but is categorical (dtype selection gives 36 'numerical' columns).

#### 1.b) Is SalePrice approximately Gaussian?

In [ ]:
# 1.b) Graphically determine whether SalePrice is approximately Gaussian.
# If it is not, suggest and apply a suitable transformation.
# Explain why considering such a transformation can be important.
#
# Guidance:
# - Use suitable graphical diagnostics (for example, a histogram and/or Q-Q plot).
# - Keep track of the transformation you choose because later questions require metrics
#   both on the transformed scale and, after inverse transformation, on the original
#   SalePrice scale.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 1.b:
#   Raw SalePrice: skew 1.88, convex Q-Q plot. log(SalePrice): skew 0.12, almost straight Q-Q line.
#   Box-Cox lambda is about -0.06 on the training data, so the plain log is the natural choice (no parameter to fit).
#   Keep the USD target too: every final metric must be reported in USD after np.exp(prediction).

**Answer 1.b**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 1.b.*

#### 1.c) Train/test split (70% / 30%)

In [ ]:
# 1.c) Split the data into training and test sets.
# Randomly assign 70% of the observations to the training set and 30% to the test set.
#
# IMPORTANT:
# - Save an UNPROCESSED copy of X_train and X_test before doing any imputation,
#   standardization, or one-hot encoding.
# - You will return to these raw splits in Question 3.b) to build a leakage-safe
#   cross-validation pipeline.
#
# Suggested naming:
# X_train_raw, X_test_raw, y_train, y_test

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 1.c:
#   train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED) -> 1022 training and 438 test houses.
#   With SEED = 42 the two 'Partial' sale outliers (Ids 524, 1299) fall in the TRAINING set, Id 692 in the test set.

#### 1.d) Missing values, standardization, one-hot encoding

In [ ]:
# 1.d) Prepare X_train and X_test using TRAINING-SET information only.
#
# Required steps:
# (i)   Numerical missing values: impute with the corresponding training-set mean.
# (ii)  Categorical missing values: impute with the most frequent category in the
#       corresponding training column.
# (iii) Treat 'NA' / 'None' as actual categorical levels where the data description
#       specifies that they are valid categories, rather than automatically treating
#       them as missing.
# (iv)  Standardize numerical features using training-set means and standard deviations,
#       and apply the same transformation, using these training-set statistics, to
#       the test set.
# (v)   One-hot encode the categorical variables.
# (vi)  Ensure that the test set contains the same encoded feature columns, in the
#       same order, as the training set.
#
# Hint:
# If pd.get_dummies is applied separately to the training and test sets, they may
# produce different dummy-variable columns because some categorical levels occur
# in only one of the two sets. The regression model must receive the same feature
# columns at training and prediction time. One possible way to align the encoded
# test columns to the training columns is:
#
# X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
#
# This creates any training dummy column missing from the test set and fills it
# with zero, while also matching the training-column order.
#
# Expected output / checks:
# - Report the final shapes of the prepared training and test feature matrices.
# - Verify that the encoded feature columns and their order are identical.
# - Do not use any test-set statistic to fit the preprocessing.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 1.d:
#   'NA' is a real level for 14 variables; for the 5 basement variables only if TotalBsmtSF == 0
#   (Ids 949 and 333 have a basement, so their 'NA' is genuinely missing). All other 'NA' are missing.
#   Impute first, then standardize with ddof=0 (the same as StandardScaler, so Question 3 can reproduce it exactly).
#   Expected with SEED = 42: train (1022, 311), test (438, 311); 5 test levels do not occur in training.

### Question 2 - Linear Regression on Numerical Features

#### 2.a) OLS on the 35 numerical features (sklearn)

In [ ]:
# 2.a) Fit an OLS regression using ONLY the numerical predictors and sklearn.
# Include an intercept.
#
# Required output:
# - A table with each numerical feature and its estimated regression coefficient.
# - In-sample and out-of-sample MSE and R^2.
# - If y was transformed in Question 1.b), report MSE and R^2:
#     * on the transformed target scale; and
#     * on the original SalePrice scale after inverse-transforming predictions.
# - Briefly explain what the two scales tell you about model performance.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.a:
#   Write ONE metrics helper (MSE and R2, in/out of sample, log scale and USD scale) and reuse it for every model.
#   The coefficients of the 8 area variables in the two exact identities are not unique (see 2.b.iv).
#   With SEED = 42 the in-sample USD MSE is LARGER than the out-of-sample one: explain it (outliers 524, 1299).

**Answer 2.a**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 2.a.*

#### 2.b) OLS with matrix algebra (training set only)

##### (i) Coefficients from the normal equation

In [ ]:
# 2.b) Reproduce and investigate OLS using matrix algebra on the TRAINING SET ONLY.
#
# Let A denote the design matrix containing the numerical predictors plus a column
# of ones for the intercept.
#
# (i) Compute the estimated coefficients using numpy.
#     Use a numerically stable linear-system solver such as np.linalg.solve(...)
#     whenever the system is numerically well behaved; do not explicitly form
#     a matrix inverse merely to solve the normal equations.
#
#     Expected output:
#     - Estimated intercept and feature coefficients.
#     - A comparison with the sklearn estimates from Question 2.a).

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.b.i:
#   np.linalg.solve raises no error here, but A has rank 34 < 36, so A'A is singular: it returns ONE of
#   infinitely many solutions. Check np.linalg.matrix_rank(A) and the condition number and say so.

##### (ii) Standard errors

In [ ]:
# (ii) Compute the standard error of each estimated coefficient using numpy.
#
#      Expected output:
#      - A table containing coefficient estimates and their standard errors,
#        including the intercept.
#      - Clearly state which estimate corresponds to the intercept.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.b.ii:
#   sigma^2 = SSR / (m - (d+1)) = SSR / 986. The SEs of the 8 area variables come out around 1e5: meaningless.

##### (iii) Check against Question 2.a

In [ ]:
# (iii) Using your matrix-algebra implementation, verify that the in-sample
#       MSE and R^2 agree with the corresponding results from Question 2.a),
#       up to numerical rounding.

##### (iv) Rank and singular values of A

In [ ]:
# (iv) Inspect the numerical structure of A.
#
#      Required diagnostics:
#      - rank of A;
#      - singular values of A.
#
#      Guidance:
#      - np.linalg.matrix_rank(...) and np.linalg.svd(..., compute_uv=False)
#        may be useful.
#      - Comment on whether A is full column rank.
#      - Inspect the smallest singular values and discuss whether they suggest
#        potential numerical instability.
#      - Relate this evidence to the OLS calculations above.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.b.iv:
#   rank 34 of 36; two singular values about 1e-14 (largest about 85); the next one is about 10, so the rest
#   of A is well conditioned. The right singular vectors of the two zero singular values show WHICH columns
#   are dependent: TotalBsmtSF = BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF and GrLivArea = 1stFlrSF + 2ndFlrSF + LowQualFinSF.

**Answer 2.b.iv**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 2.b.iv.*

##### (v) Moore-Penrose pseudoinverse

In [ ]:
# (v) Replace the usual inverse-based expression by the Moore-Penrose pseudoinverse.
#     Use np.linalg.pinv(...) with its default cutoff.
#
#     Required comparison:
#     - Do the coefficient estimates change?
#     - Does the residual variance estimate change?
#     - Do the standard errors change?
#     - Explain when the ordinary inverse-based and pseudoinverse-based results
#       should agree and when they may differ, using the diagnostics from (iv).

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.b.v:
#   pinv cuts singular values below 1e-15 * s_max, so it returns the minimum-norm solution.
#   Same fitted values, same sigma^2, same SEs for the 28 other coefficients; different, finite values for the 8 area ones.

**Answer 2.b.v**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 2.b.v.*

##### (vi) statsmodels

In [ ]:
# (vi) Confirm your results with statsmodels OLS.
#
#      Required output:
#      - Coefficient estimates and standard errors from statsmodels.
#      - Compare them with your matrix-algebra results up to numerical rounding.
#      - If they differ, relate the discrepancy to the rank / numerical-stability
#        diagnostics above.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 2.b.vi:
#   statsmodels uses pinv (same coefficients as (v)) but df_resid = m - rank(A) = 988 instead of 986:
#   every SE is smaller by the factor sqrt(986/988) = 0.99899. Its summary also warns about a tiny eigenvalue.

**Answer 2.b.vi**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 2.b.vi.*

### Minimal sklearn Workflow Example (Illustration Only)

In [ ]:
# This example demonstrates the WORKFLOW required in Question 3.b):
#
# raw data
#   -> ColumnTransformer
#   -> Pipeline
#   -> GridSearchCV
#   -> best hyperparameters / CV score
#   -> refitted best estimator
#   -> held-out test evaluation
#
# It uses a small artificial dataset and Ridge regression. It is NOT a solution
# to the housing problem, and its hyperparameter grid should not be copied blindly.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.metrics import mean_squared_error

# ----- 1. Create a tiny mixed-type example dataset -----
rng = np.random.default_rng(1)
n = 60

X_example = pd.DataFrame({
    "size": rng.normal(100, 15, n),
    "age": rng.normal(20, 7, n),
    "region": rng.choice(["A", "B", "C"], size=n),
})

# Add a few missing values only to illustrate preprocessing.
X_example.loc[[2, 11, 31], "size"] = np.nan
X_example.loc[[5, 27], "region"] = np.nan

# Construct an arbitrary regression target.
region_effect = X_example["region"].map({"A": 0.0, "B": 5.0, "C": -3.0}).fillna(0.0)
y_example = (
    2.0 * X_example["size"].fillna(X_example["size"].mean())
    - 1.5 * X_example["age"]
    + region_effect
    + rng.normal(0, 5, n)
)

X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_example, y_example, test_size=0.25, random_state=1
)

# ----- 2. Define preprocessing -----
numeric_features = ["size", "age"]
categorical_features = ["region"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_example = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# ----- 3. Put preprocessing and the model in ONE pipeline -----
pipeline_example = Pipeline([
    ("preprocessor", preprocessor_example),
    ("model", Ridge()),
])

# Pipeline parameter names use step__parameter.
param_grid_example = {
    "model__alpha": [0.1, 1.0, 10.0],
}

# Define the CV folds once. The same idea can be used to ensure fair model comparisons.
cv_example = KFold(n_splits=4, shuffle=True, random_state=1)

search_example = GridSearchCV(
    estimator=pipeline_example,
    param_grid=param_grid_example,
    scoring="neg_mean_squared_error",
    cv=cv_example,
    refit=True,
)

# IMPORTANT:
# During each CV split, the preprocessing steps are fitted only on that fold's
# training observations because they are inside the Pipeline.
search_example.fit(X_ex_train, y_ex_train)

# ----- 4. Inspect the selected configuration and its CV performance -----
best_idx = search_example.best_index_
cv_mean_mse = -search_example.cv_results_["mean_test_score"][best_idx]
cv_std_mse = search_example.cv_results_["std_test_score"][best_idx]

print("Best parameters:", search_example.best_params_)
print(f"Mean CV MSE: {cv_mean_mse:.3f}")
print(f"Std. CV MSE: {cv_std_mse:.3f}")

# ----- 5. Final held-out test evaluation -----
# With refit=True (the default), best_estimator_ has already been refitted on ALL
# of X_ex_train using the selected hyperparameters.
best_model_example = search_example.best_estimator_
test_pred_example = best_model_example.predict(X_ex_test)
test_mse_example = mean_squared_error(y_ex_test, test_pred_example)

print(f"Test MSE: {test_mse_example:.3f}")

# Things to understand before Question 3.b):
# - Why must the preprocessing be inside the Pipeline during CV?
# - Why is the parameter called "model__alpha"?
# - What does refit=True do after CV selects the best hyperparameters?
# - Why should the held-out test set be used only after model selection?

# Visualize the structure of the pipeline
pipeline_example


### Question 3 - Regularization Techniques

#### 3.a) OLS on all prepared features

In [ ]:
# 3.a) Fit OLS using ALL prepared explanatory variables (numerical + categorical).
#
# Use the prepared Housing dataset from Question 1.d).
#
# Required output:
# - In-sample and out-of-sample MSE and R^2 on the ORIGINAL SalePrice scale.
# - If the target was transformed, inverse-transform predictions before computing
#   these metrics.
# - Compare the results with the regression using only numerical features from
#   Question 2.a).
# - Does including the categorical features improve out-of-sample predictive
#   performance?
# - How does the gap between in-sample and out-of-sample performance change?

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 3.a:
#   The full design (1022 x 312 with intercept) has rank 254: 58 exact dependencies (one-hot blocks, identical
#   'no garage'/'no basement' dummies, the 2 area identities). Expect a much better in-sample fit but NO
#   out-of-sample gain over 2.a: rare levels seen in 1-3 training houses get huge coefficients (overfitting).

**Answer 3.a**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.a.*

#### 3.b) Truncated pseudoinverse, Ridge, Lasso and Elastic Net with 8-fold CV

In [ ]:
# 3.b) Implement and tune:
# - Truncated Pseudoinverse regression;
# - Ridge;
# - Lasso;
# - Elastic Net.
#
# Use 8-fold CV, MSE as the model-selection criterion, and the SAME CV folds
# for all four methods.
#
# IMPORTANT: Start from the UNPROCESSED X_train_raw / X_test_raw saved in 1.c).
# The preprocessing must be INSIDE the Pipeline so that all learned preprocessing
# quantities are estimated separately within each CV training fold.
#
# Follow the workflow below.

##### 3.b.i) One reusable preprocessing component

In [ ]:
# 3.b.i) Build ONE reusable preprocessing component.
#
# It should reproduce the rules from Question 1.d):
# - numerical imputation;
# - numerical standardization;
# - categorical imputation;
# - one-hot encoding;
# - correct treatment of 'NA' / 'None' where these are valid categorical levels.
#
# Make sure a category that is absent from a particular CV training fold does not
# cause an error when its validation fold is transformed.
#
# Hint: OneHotEncoder has an option for handling categories not seen during fit.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 3.b.i:
#   FunctionTransformer(recode_missing, feature_names_out='one-to-one') -> ColumnTransformer(num: mean imputer +
#   StandardScaler; cat: most_frequent imputer + OneHotEncoder(handle_unknown='ignore', sparse_output=False)).
#   Check that it reproduces the 1.d matrices exactly (it does: max difference 0.0 in the preparation check).

##### 3.b.ii) One pipeline per method

In [ ]:
# 3.b.ii) Construct one Pipeline for each method using the SAME preprocessor.
#
# Required structures:
#
# Ridge / Lasso / Elastic Net:
# raw data -> preprocessor -> model
#
# Truncated Pseudoinverse:
# raw data -> preprocessor -> TruncatedSVD -> LinearRegression
#
# All regressions should contain an intercept, and the intercept should NOT be
# regularized.
#
# Do not implement a custom TPI estimator: use TruncatedSVD as the dimensionality-
# reduction step before LinearRegression.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 3.b.ii:
#   TruncatedSVD(algorithm='arpack', random_state=SEED): exact and reproducible (the default 'randomized' is approximate).
#   The folds have 305-310 features after one-hot and rank 246-251, so n_components must stay below about 245.

##### 3.b.iii) Hyperparameter grids and tuning

In [ ]:
# 3.b.iii) Define reasonable hyperparameter grids and tune with GridSearchCV.
#
# Tune:
# - regularization strength for Ridge;
# - regularization strength for Lasso;
# - regularization strength and mixing parameter for Elastic Net;
# - number of retained singular directions for Truncated Pseudoinverse.
#   With sklearn's TruncatedSVD, this is n_components: the number of retained
#   singular components, not a singular-value cutoff.
#
# Pipeline parameters follow the step__parameter convention.
# Examples of the NAMING convention (not recommended parameter values):
# - model__alpha
# - model__l1_ratio
# - svd__n_components
#
# Define ONE 8-fold CV splitter and reuse it for every method so the comparison
# uses exactly the same folds.
#
# Use MSE as the selection criterion. sklearn commonly expresses this as
# scoring="neg_mean_squared_error" because GridSearchCV maximizes scores.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 3.b.iii:
#   Grid ranges that worked in the scans: Ridge alpha 1e-3..1e4; Lasso alpha 1e-6..1e-1;
#   Elastic Net alpha 1e-5..1 x l1_ratio 0.01..1; TPI n_components 10..245. Create the KFold object ONCE.

##### 3.b.iv) Are the selected values at the edge of a grid?

In [ ]:
# 3.b.iv) Inspect the selected hyperparameters.
#
# For EACH method:
# - Check whether the selected value lies at or near a boundary of the grid.
# - If it does, extend the relevant range and repeat the search.
# - Briefly justify the final grid / range you report.
#
# Do not stop automatically at the first grid if the optimum appears constrained
# by the range you chose.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step 3.b.iv:
#   Write a small helper that reports, for each parameter, whether the best value is the first or last grid
#   point, and plot the mean CV MSE against each hyperparameter (a U-shape shows the minimum is inside).

**Answer 3.b.iv**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.b.iv.*

##### 3.b.v) Final evaluation of the refitted models

In [ ]:
# 3.b.v) Evaluate the selected and refitted models.
#
# With refit=True, GridSearchCV refits the selected pipeline on the COMPLETE outer
# training set after cross-validation.
#
# Required final metrics:
# - in-sample MSE and R^2 on the ORIGINAL SalePrice scale;
# - out-of-sample MSE and R^2 on the ORIGINAL SalePrice scale.
#
# If y was transformed, inverse-transform predictions before computing these
# final train/test metrics.
#
# Compare the four regularized methods with BOTH OLS benchmarks from 2.a) and 3.a).

##### Summary table

In [ ]:
# 3.b) REQUIRED SUMMARY TABLE
#
# Include the four regularized models and the OLS benchmarks from 2.a) and 3.a).
# For quantities that do not apply to OLS, use "--".
#
# Suggested columns:
# Model
# Selected hyperparameter(s)
# Mean CV MSE
# Std. CV MSE
# In-sample MSE
# In-sample R^2
# Out-of-sample MSE
# Out-of-sample R^2
#
# For each regularized model, the CV MSE is the mean validation MSE across the
# 8 folds for the SELECTED hyperparameter configuration.
#
# Hint:
# - search.best_index_ identifies the selected row in search.cv_results_.
# - search.cv_results_["mean_test_score"][search.best_index_] gives the selected
#   mean CV score.
# - search.cv_results_["std_test_score"][search.best_index_] gives its standard
#   deviation across validation folds.
# - If scoring="neg_mean_squared_error", reverse the sign of mean_test_score when
#   reporting MSE. Do NOT reverse the sign of the standard deviation.
#
# If SalePrice was transformed, the CV MSE used for model selection is on the
# transformed target scale. The final in-sample / out-of-sample metrics above
# should be reported on the original SalePrice scale.

**Answer 3.b.v**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.b.v.*

##### Why is the intercept not penalized?

In [ ]:
# Why is it important that the intercept is not penalized during regularization?

**Answer 3.b (intercept)**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.b (intercept).*

#### 3.c) Pipeline parameter names and `refit=True`

In [ ]:
# 3.c) Pipeline parameter notation and refitting.
#
# Explain:
# - What a pipeline parameter name such as model__alpha means.
# - What refit=True in GridSearchCV does after the best hyperparameters have
#   been selected.
# - Why this refitting step is useful before the final evaluation on the
#   held-out test set.

**Answer 3.c**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.c.*

#### 3.d) Sparsity versus truncation of singular directions

In [ ]:
# 3.d) Sparsity versus singular-direction truncation.
#
# Required output / discussion:
# - Number of non-zero FEATURE coefficients for Lasso (exclude the intercept).
# - Number of non-zero FEATURE coefficients for Elastic Net (exclude the intercept).
# - Compare with Ridge, which generally shrinks coefficients without setting them
#   exactly to zero.
# - For Truncated Pseudoinverse, report the SELECTED NUMBER OF SINGULAR DIRECTIONS,
#   rather than counting non-zero regression coefficients.
# - Explain why truncating singular directions is conceptually different from
#   coefficient sparsity in Lasso / Elastic Net.

**Answer 3.d**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.d.*

#### 3.e) Final model recommendation

In [ ]:
# 3.e) Final model recommendation.
#
# Which model would you recommend for predicting house prices?
#
# Support the recommendation using BOTH:
# - empirical performance; and
# - the structure of the problem.
#
# Relevant considerations may include:
# - number of features;
# - categorical variables / one-hot encoding;
# - collinearity;
# - sparsity;
# - interpretability;
# - stability / robustness;
# - nonlinearity and other limitations of linear models.
#
# Explain how the strengths and limitations of your selected method align with
# the problem structure.

**Answer 3.e**

*To be written. The key points and the numbers to check are in `helpers/Analysis/4_Plan_and_Decisions.md`, step 3.e.*

### Additional robustness checks

Optional. These give the evidence for the *Robustness* part of the report.

In [ ]:
# R.1) The pipeline preprocessor reproduces Question 1.d exactly (no hidden differences).
# R.2) Mean CV MSE against each hyperparameter (grid adequacy).
# R.3) Repeat split + tuning for several random seeds: is the ranking of the models stable?
# R.4) Influence of the unusual houses (Ids 524, 1299, 692): largest errors, fold-wise CV MSE.

# PREP NOTE (delete before submission) - helpers/Analysis/4_Plan_and_Decisions.md, step R:
#   Keep this section short: a few lines of code and one sentence per check.

### Project Report

At the end of the notebook, provide a concise synthesis of your analysis. Do not simply repeat every answer above.

#### Approach
Briefly summarize the preprocessing and modeling workflow.

#### Model selection
Explain how the models and hyperparameters were selected, including the cross-validation procedure.

#### Key results
Summarize the most important quantitative results.

#### Interpretation
Explain what the results imply about the regression methods and the housing problem.

#### Robustness
Discuss relevant checks such as leakage prevention, numerical stability, and hyperparameter-grid adequacy.

#### Limitations
State the main limitations of the analysis and models.